# Train a Dataset Builder zip

Trains yolov8n-seg on this dataset and gives back `best.onnx`.

1. Set **Runtime → Change runtime type → GPU**.
2. In the **Dataset zip** cell paste the Google Drive link of **your dataset zip** (shared as *Anyone with the link*): Colab downloads it far faster than an upload. Left empty, the cell asks you to upload the zip instead.
3. Run the cells in order, or **Runtime → Run all**.

In [ ]:
!pip install -q ultralytics

In [ ]:
#@title Dataset zip
#@markdown A Google Drive link to the zip, shared as Anyone with the link. Left empty, the zip is uploaded from this computer.
drive_link = ""  #@param {type:"string"}
import os, re
from google.colab import files
if drive_link.strip():
    import gdown
    zip_name = gdown.download(drive_link.strip(), "/content/dataset.zip", fuzzy=True)
    assert zip_name, "Drive did not give the zip: share it as Anyone with the link, or leave the link empty to upload it."
    found = re.search(r"(?:/d/|id=)([-\w]+)", drive_link)
    zip_key = found.group(1) if found else "drive"
else:
    up = files.upload()  # pick your dataset zip
    zip_name = next(iter(up))
    zip_key = zip_name.rsplit(".", 1)[0]
# The zip's runs: its Drive file or its name, and its size, so another zip with the same name gets its own.
zip_key = f"{zip_key}-{os.path.getsize(zip_name)}"
!rm -rf /content/ds && mkdir -p /content/ds && unzip -q "{zip_name}" -d /content/ds
!cat /content/ds/data.yaml

In [ ]:
#@title Keep the runs on Google Drive
#@markdown Trains into My Drive / Dataset Builder / runs, so a session that stops resumes where it left off: Run all again. Colab asks for access to Drive.
keep_on_drive = True  #@param {type:"boolean"}
import torch
from ultralytics import YOLO
runs = "/content/runs"
if keep_on_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    runs = "/content/drive/MyDrive/Dataset Builder/runs/" + zip_key

def train(stage, start, **args):
    """Resumes the stage from its last.pt, skips it once it finished (its last.pt then has epoch -1), or starts it from start."""
    last = f"{runs}/{stage}/weights/last.pt"
    if os.path.exists(last):
        try:
            epoch = torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1)
        except Exception as e:  # a save the stopped session cut short: the stage starts again
            print(stage, "last.pt unreadable, starting again:", e)
            epoch = None
        if epoch == -1:
            print(stage, "finished already: skipped")
            return
        if epoch is not None:
            YOLO(last).train(resume=True)
            return
    YOLO(start).train(data="/content/ds/data.yaml", imgsz=512, batch=0.7, cache="ram", fliplr=0.0, mosaic=1.0, project=runs, name=stage, exist_ok=True, **args)

In [ ]:
train("seg", "yolov8n-seg.pt", epochs=200, patience=20)

In [ ]:
# Fine-tune from the best weights at a lower learning rate. optimizer="auto" would ignore lr0, so it is named.
train("seg-tune", runs + "/seg/weights/best.pt", epochs=40, patience=10, optimizer="AdamW", lr0=0.0002, lrf=0.1, cos_lr=True, warmup_epochs=1)

In [ ]:
# The fine-tune is kept only if it scores better on the valid split: it does not always help.
first = YOLO(runs + "/seg/weights/best.pt").val(data="/content/ds/data.yaml", imgsz=512).fitness
second = YOLO(runs + "/seg-tune/weights/best.pt").val(data="/content/ds/data.yaml", imgsz=512).fitness
run = runs + ("/seg-tune/weights/" if second > first else "/seg/weights/")
print("fine-tune", "kept" if second > first else "dropped", round(first, 4), "->", round(second, 4))
best = YOLO(run + "best.pt")
best.val(data="/content/ds/data.yaml", split="test", imgsz=512)
best.export(format="onnx", imgsz=512, opset=12, simplify=True)
# Downloaded from a copy: the browser fetches the files later, after Save to Google Drive may have removed the runs.
import shutil
out = "/content/best/"
os.makedirs(out, exist_ok=True)
for weights in ("best.pt", "best.onnx"):
    shutil.copy(run + weights, out)
files.download(out + "best.onnx")
files.download(out + "best.pt")

In [ ]:
#@title Save to Google Drive
#@markdown Copies best.pt and best.onnx to My Drive / Dataset Builder, so they outlive this session, then removes the runs kept on Drive. Colab asks for access to Drive.
save_to_drive = True  #@param {type:"boolean"}
if save_to_drive:
    import os, shutil, time
    from google.colab import drive
    drive.mount("/content/drive")
    kept = "/content/drive/MyDrive/Dataset Builder/" + time.strftime("seg-%Y%m%d-%H%M")
    os.makedirs(kept, exist_ok=True)
    for weights in ("best.pt", "best.onnx"):
        shutil.copy(out + weights, kept)
    print("saved to", kept)
    # Once the weights are kept, the runs on Drive are only clutter: Drive stays clean.
    if keep_on_drive:
        shutil.rmtree(runs, ignore_errors=True)
        print("removed", runs)

The ONNX keeps no class names: they are `names` in `data.yaml`, in that order.

A session that stopped resumes with **Runtime → Run all** and the same zip: each training goes on from its `last.pt` on Drive, and one that finished is skipped. With *Keep the runs on Google Drive* off, the runs are in `/content/runs` and go with the session.

`best.pt` trains on later, with more epochs or new images: upload it, then `YOLO("best.pt").train(data=..., epochs=...)`.